# UCS420 Assignment 9 — Landing AI (Vision-based Cognitive System)
**Name:** Aastha Tyagi  
**Roll No.:** 1024160131  

Two assistants:
1. **MediLabel** — reads a medicine-bottle label (OCR) and answers safety questions.
2. **MoodMate** — detects a facial expression and replies in a matching tone.

Run in Google Colab (GPU optional). The first code cell installs packages.


In [ ]:
# Colab: uncomment if pytesseract / opencv are missing
# !apt-get -qq update && apt-get -qq install -y tesseract-ocr
# %pip install -q pytesseract opencv-python-headless pillow matplotlib

from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import matplotlib.pyplot as plt

OUT = Path("assignment9_assets")
OUT.mkdir(exist_ok=True)
print("asset folder:", OUT.resolve())


## Part A — Cognitive assistant for medicine-bottle labels

Synthetic labels are generated so the notebook does not depend on a Kaggle login. Each label contains drug name, strength, directions, batch and expiry — the same fields a real OTC bottle shows. OCR is Tesseract; a small parser turns the text into a structured card the assistant can query.


In [ ]:
def make_label(path, title, lines, color=(18, 72, 140)):
    img = Image.new("RGB", (720, 420), (245, 245, 240))
    draw = ImageDraw.Draw(img)
    draw.rectangle([0, 0, 720, 70], fill=color)
    try:
        font_b = ImageFont.truetype("arial.ttf", 28)
        font = ImageFont.truetype("arial.ttf", 20)
        font_s = ImageFont.truetype("arial.ttf", 16)
    except OSError:
        font_b = font = font_s = ImageFont.load_default()
    draw.text((24, 18), title, fill="white", font=font_b)
    y = 90
    for line in lines:
        draw.text((24, y), line, fill=(20, 20, 20), font=font)
        y += 36
    draw.text((24, 380), "For educational OCR demo — not a real medicine", fill=(90, 90, 90), font=font_s)
    img.save(path)
    return img

labels = {
    "metformin": dict(
        title="GLUCOFORM 500 mg",
        lines=[
            "Generic: Metformin Hydrochloride Tablets",
            "Strength: 500 mg",
            "Take 1 tablet twice daily after meals",
            "Batch: MX-4412   Exp: 12/2027",
            "Store below 30 C. Keep out of reach of children.",
            "Warning: Do not use with excess alcohol.",
        ],
    ),
    "amoxicillin": dict(
        title="AMOXICARE 250 mg",
        lines=[
            "Generic: Amoxicillin Capsules",
            "Strength: 250 mg",
            "Take 1 capsule every 8 hours for 5 days",
            "Batch: AC-1188   Exp: 03/2026",
            "Complete the full course even if you feel better.",
            "Allergy warning: penicillin group.",
        ],
    ),
    "paracetamol": dict(
        title="FEVEREASE 650 mg",
        lines=[
            "Generic: Paracetamol Tablets",
            "Strength: 650 mg",
            "Take 1 tablet every 6 hours if needed",
            "Batch: PC-9001   Exp: 08/2028",
            "Do not exceed 4 tablets in 24 hours.",
            "Warning: Overdose can damage the liver.",
        ],
    ),
}

label_images = {}
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (key, spec) in zip(axes, labels.items()):
    path = OUT / f"{key}_label.png"
    im = make_label(path, spec["title"], spec["lines"])
    label_images[key] = path
    ax.imshow(im)
    ax.set_title(key)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
import re

try:
    import pytesseract
    TESS = True
except Exception as e:
    TESS = False
    print("pytesseract not importable:", e)


def ocr_image(path):
    if not TESS:
        # Fallback: read the known lines so the assistant still demos offline.
        # In Colab with tesseract installed this branch is not used.
        key = Path(path).stem.replace("_label", "")
        return "\n".join([labels[key]["title"], *labels[key]["lines"]])
    img = Image.open(path)
    return pytesseract.image_to_string(img)


def parse_label(text):
    info = {"raw": text.strip()}
    m = re.search(r"Generic:\s*(.+)", text)
    info["generic"] = m.group(1).strip() if m else None
    m = re.search(r"Strength:\s*(.+)", text)
    info["strength"] = m.group(1).strip() if m else None
    m = re.search(r"Batch:\s*(\S+)", text)
    info["batch"] = m.group(1).strip() if m else None
    m = re.search(r"Exp:\s*(\S+)", text)
    info["expiry"] = m.group(1).strip() if m else None
    info["warnings"] = [ln.strip() for ln in text.splitlines() if re.search(r"warning|do not|allergy|overdose", ln, re.I)]
    info["directions"] = [ln.strip() for ln in text.splitlines() if re.search(r"take |course", ln, re.I)]
    return info

cards = {k: parse_label(ocr_image(p)) for k, p in label_images.items()}
for k, c in cards.items():
    print("=" * 60, k)
    print(c["raw"])
    print("PARSED:", {kk: c[kk] for kk in c if kk != "raw"})


In [ ]:
def medilabel_ask(bottle, question):
    card = cards[bottle]
    q = question.lower()
    if "name" in q or "generic" in q:
        ans = card["generic"]
    elif "strength" in q or "dose" in q or "mg" in q:
        ans = card["strength"]
    elif "expir" in q:
        ans = card["expiry"]
    elif "batch" in q:
        ans = card["batch"]
    elif "warn" in q or "alcohol" in q or "allerg" in q:
        ans = " | ".join(card["warnings"]) or "No warning line parsed."
    elif "how" in q or "take" in q or "direction" in q:
        ans = " | ".join(card["directions"]) or "No direction line parsed."
    else:
        ans = "I can answer name, strength, expiry, batch, directions or warnings for this bottle."
    return (
        f"[MediLabel] bottle={bottle}\n{ans}\n"
        "Educational OCR demo only — confirm with the physical pack and a pharmacist."
    )

demo_qs = [
    ("metformin", "What is the generic name?"),
    ("metformin", "Any warning about alcohol?"),
    ("amoxicillin", "How should I take it?"),
    ("paracetamol", "When does it expire?"),
]
for b, q in demo_qs:
    print(q)
    print(medilabel_ask(b, q))
    print()


## Part B — Emotion-aware cognitive assistant

Sample face crops are drawn as simple expression cues (eyebrows + mouth) so the notebook runs without downloading a face dataset. A tiny rule-based vision step measures mouth curvature and eyebrow gap — the same features a CNN would use — then the assistant changes its wording. In Colab you can replace `detect_expression()` with `fer.FER`.


In [ ]:
def draw_face(path, mood):
    img = Image.new("RGB", (240, 240), (255, 224, 196))
    d = ImageDraw.Draw(img)
    d.ellipse([20, 20, 220, 220], fill=(255, 224, 196), outline=(80, 50, 40), width=3)
    d.ellipse([70, 90, 100, 120], fill=(40, 40, 40))
    d.ellipse([140, 90, 170, 120], fill=(40, 40, 40))
    if mood == "happy":
        d.arc([70, 120, 170, 190], 20, 160, fill=(180, 40, 40), width=6)
        d.line([60, 70, 100, 75], fill=(40, 40, 40), width=4)
        d.line([140, 75, 180, 70], fill=(40, 40, 40), width=4)
    elif mood == "sad":
        d.arc([70, 150, 170, 210], 200, 340, fill=(40, 40, 140), width=6)
        d.line([60, 80, 100, 70], fill=(40, 40, 40), width=4)
        d.line([140, 70, 180, 80], fill=(40, 40, 40), width=4)
    elif mood == "angry":
        d.line([70, 170, 170, 170], fill=(120, 0, 0), width=6)
        d.line([55, 70, 105, 85], fill=(40, 40, 40), width=5)
        d.line([135, 85, 185, 70], fill=(40, 40, 40), width=5)
    else:
        d.line([80, 170, 160, 170], fill=(40, 40, 40), width=5)
        d.line([60, 75, 100, 75], fill=(40, 40, 40), width=4)
        d.line([140, 75, 180, 75], fill=(40, 40, 40), width=4)
    img.save(path)
    return img

moods = ["happy", "sad", "angry", "neutral"]
face_paths = {}
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, mood in zip(axes, moods):
    p = OUT / f"face_{mood}.png"
    im = draw_face(p, mood)
    face_paths[mood] = p
    ax.imshow(im)
    ax.set_title(mood)
    ax.axis("off")
plt.show()


In [ ]:
def detect_expression(path):
    """Geometric stand-in for a CNN emotion classifier (FER in Colab)."""
    arr = np.array(Image.open(path).convert("L"))
    # mouth band
    mouth = arr[150:200, 70:170]
    upper, lower = mouth[:15].mean(), mouth[-15:].mean()
    brow = arr[60:90, 50:190].mean()
    if lower + 8 < upper:
        return "happy", 0.82
    if upper + 8 < lower:
        return "sad", 0.80
    if brow < 170:
        return "angry", 0.75
    return "neutral", 0.70

RESPONSES = {
    "happy": "You look upbeat. I can keep the explanation short and cheerful.",
    "sad": "You look low. I will go slowly and keep the language gentle.",
    "angry": "You look frustrated. I will stay direct and skip extra chatter.",
    "neutral": "Neutral expression detected. I will use a plain informational tone.",
}

print("Expression detection on sample faces")
for mood, path in face_paths.items():
    pred, conf = detect_expression(path)
    print(f" drawn={mood:7}  predicted={pred:7}  conf={conf:.2f}  -> {RESPONSES[pred]}")


def moodmate(path, user_text):
    pred, conf = detect_expression(path)
    return f"[{pred} | {conf:.2f}] {RESPONSES[pred]}\nReply: I heard '{user_text}'. Let us handle it step by step."

print("\n", moodmate(face_paths["sad"], "I cannot understand this label"))


### Optional Colab upgrade (real FER)

```python
# %pip install -q fer
# from fer import FER
# detector = FER(mtcnn=True)
# detector.detect_emotions(cv2.imread("your_face.jpg"))
```

Use a webcam snapshot or any FER-2013 image. The assistant API above stays the same: map the top emotion to `RESPONSES`.
